# 03 - Business Analysis
### Google Play Store App Analytics

This notebook turns the EDA and statistical findings into concrete business
questions and data-driven recommendations. All conclusions are backed by
numbers computed from the actual cleaned dataset -- nothing here is fabricated.

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/processed/googleplaystore_cleaned.csv')
df.shape

(9659, 23)

## Statistical Analysis: Correlations

In [2]:
pairs = [
    ('rating','installs'), ('rating','reviews'), ('price','installs'),
    ('price','rating'), ('size_mb','installs'), ('reviews','installs'),
]
rows = []
for a, b in pairs:
    sub = df[[a,b]].dropna()
    pr, pp = stats.pearsonr(sub[a], sub[b])
    sr, sp = stats.spearmanr(sub[a], sub[b])
    rows.append({'pair': f'{a} vs {b}', 'n': len(sub), 'pearson_r': round(pr,4),
                 'pearson_p': round(pp,6), 'spearman_r': round(sr,4), 'spearman_p': round(sp,6)})
corr_df = pd.DataFrame(rows)
corr_df

,pair,n,pearson_r,pearson_p,spearman_r,spearman_p
0,rating vs installs,8196,0.0402,0.000268,0.0272,0.01385
1,rating vs reviews,8196,0.0552,0.000001,0.1196,0.00000
2,price vs installs,9659,-0.0094,0.355075,-0.2316,0.00000
3,price vs rating,8196,-0.0211,0.056015,0.0705,0.00000
4,size_mb vs installs,8431,0.1342,0.000000,0.3101,0.00000
5,reviews vs installs,9659,0.6251,0.000000,0.9677,0.00000


**Note on correlation vs causation:** these coefficients describe the strength
of association only. For example, `reviews` and `installs` are strongly
correlated (Pearson r ≈ 0.63, Spearman r ≈ 0.97), but that does not mean
reviews *cause* installs -- both are more plausibly driven by a third factor:
an app's overall popularity and organic reach.

## Statistical Analysis: Free vs Paid

In [3]:
free_rating = df.loc[df['type']=='Free','rating'].dropna()
paid_rating = df.loc[df['type']=='Paid','rating'].dropna()
free_installs = df.loc[df['type']=='Free','installs'].dropna()
paid_installs = df.loc[df['type']=='Paid','installs'].dropna()

t_stat, t_p = stats.ttest_ind(free_rating, paid_rating, equal_var=False)
u_stat, u_p = stats.mannwhitneyu(free_installs, paid_installs, alternative='two-sided')

print(f'Free avg rating: {free_rating.mean():.3f} | Paid avg rating: {paid_rating.mean():.3f}')
print(f"Welch t-test p-value: {t_p:.6f} -> {'significant' if t_p < 0.05 else 'not significant'} difference")
print()
print(f'Free median installs: {int(free_installs.median()):,} | Paid median installs: {int(paid_installs.median()):,}')
print(f"Mann-Whitney U p-value: {u_p:.3e} -> {'significant' if u_p < 0.05 else 'not significant'} difference")

Free avg rating: 4.166 | Paid avg rating: 4.262
Welch t-test p-value: 0.000050 -> significant difference

Free median installs: 100,000 | Paid median installs: 1,000
Mann-Whitney U p-value: 7.959e-116 -> significant difference


## Business Question 1: What categories should a new app developer target?

Balancing high demand (installs) against competition (app count) and
achievable quality (avg rating).

In [4]:
cat_stats = df.groupby('category').agg(
    app_count=('app','count'), avg_installs=('installs','mean'),
    avg_rating=('rating','mean'), total_installs=('installs','sum')
).reset_index()

# "Blue ocean" categories: below-median competition, above-median demand
median_count = cat_stats['app_count'].median()
median_installs = cat_stats['avg_installs'].median()
opportunity = cat_stats[(cat_stats['app_count'] < median_count) & (cat_stats['avg_installs'] > median_installs)]
opportunity.sort_values('avg_installs', ascending=False)

,category,app_count,avg_installs,avg_rating,total_installs
31,VIDEO_PLAYERS,164,2.397502e+07,4.044966,3931902720
9,ENTERTAINMENT,86,1.144953e+07,4.129070,984660000
26,SHOPPING,202,6.932420e+06,4.230556,1400348785
32,WEATHER,79,4.570893e+06,4.243056,361100520
19,MAPS_AND_NAVIGATION,131,3.841846e+06,4.036441,503281890


## Business Question 2: Should developers build free or paid apps?

In [5]:
print(f"Free apps: {(df['type']=='Free').mean()*100:.1f}% of the store")
print(f"Free avg installs: {df.loc[df['type']=='Free','installs'].mean():,.0f}")
print(f"Paid avg installs: {df.loc[df['type']=='Paid','installs'].mean():,.0f}")
print()
print("Conclusion: free apps dominate both volume and installs by a wide, statistically")
print("significant margin. Paid apps rate slightly higher on average, suggesting users")
print("who pay upfront tend to be more satisfied -- but reach far fewer people.")

Free apps: 92.2% of the store
Free avg installs: 8,452,012
Paid avg installs: 76,079

Conclusion: free apps dominate both volume and installs by a wide, statistically
significant margin. Paid apps rate slightly higher on average, suggesting users
who pay upfront tend to be more satisfied -- but reach far fewer people.


## Business Question 3: Does price negatively affect installs?

In [6]:
price_bucket_order = ['Free','$0.01-$1','$1-$5','$5-$10','$10+']
price_installs = df.groupby('price_bucket', observed=True)['installs'].mean().reindex(price_bucket_order)
print(price_installs)
print()
sub = df[['price','installs']].dropna()
r, p = stats.spearmanr(sub['price'], sub['installs'])
print(f'Spearman correlation price vs installs: {r:.4f} (p={p:.2e})')
print('A negative, statistically significant correlation confirms: higher price is')
print('associated with meaningfully fewer installs, though the relationship is not perfectly linear')
print('($5-$10 apps slightly outperform $0.01-$1 apps, suggesting a small quality-signaling effect).')

price_bucket
Free        8.452012e+06
$0.01-$1    1.310667e+05
$1-$5       4.953371e+04
$5-$10      1.767790e+05
$10+        1.199777e+04
Name: installs, dtype: float64

Spearman correlation price vs installs: -0.2316 (p=9.15e-118)
A negative, statistically significant correlation confirms: higher price is
associated with meaningfully fewer installs, though the relationship is not perfectly linear
($5-$10 apps slightly outperform $0.01-$1 apps, suggesting a small quality-signaling effect).


## Business Question 4: Which categories have high installs but relatively poor ratings?

In [7]:
low_rating_high_installs = cat_stats[cat_stats['avg_rating'] < 4.0].sort_values('avg_installs', ascending=False)
low_rating_high_installs

,category,app_count,avg_installs,avg_rating,total_installs
7,DATING,170,828971.217647,3.980451,140925107


## Business Question 5: What characteristics are common among highly successful free apps?

In [8]:
successful = df[(df['rating'] >= 4.5) & (df['installs'] >= 1_000_000) & (df['type'] == 'Free')]
print(f'{len(successful)} apps meet this bar (rating >= 4.5, installs >= 1M, free)')
successful[['app','category','rating','installs','reviews']].sort_values('installs', ascending=False).head(15)

975 apps meet this bar (rating >= 4.5, installs >= 1M, free)


,app,category,rating,installs,reviews
22,Google Photos,PHOTOGRAPHY,4.5,1000000000,10859051
6,Subway Surfers,GAME,4.5,1000000000,27725352
2,Instagram,SOCIAL,4.5,1000000000,66577446
192,Google Duo - High Quality Video Calls,COMMUNICATION,4.6,500000000,2083237
11,UC Browser - Fast Download Private & Secure,COMMUNICATION,4.5,500000000,17714850
8,"Security Master - Antivirus, VPN, AppLock, Boo...",TOOLS,4.7,500000000,24900999
191,Microsoft Word,PRODUCTIVITY,4.5,500000000,2084126
14,My Talking Tom,GAME,4.5,500000000,14892469
53,MX Player,VIDEO_PLAYERS,4.5,500000000,6474672
44,SHAREit - Transfer & Share,TOOLS,4.6,500000000,7790693


## Business Question 6: Which content-rating segments perform best?

In [9]:
content_perf = df.groupby('content_rating').agg(
    app_count=('app','count'), avg_rating=('rating','mean'), avg_installs=('installs','mean')
).sort_values('avg_installs', ascending=False)
content_perf

,app_count,avg_rating,avg_installs
content_rating,,,
Teen,1036,4.225658,1.591436e+07
Everyone 10+,322,4.225902,1.247289e+07
Everyone,7903,4.166349,6.627729e+06
Mature 17+,393,4.121849,6.203529e+06
Adults only 18+,3,4.300000,6.666667e+05
Unrated,2,4.100000,2.525000e+04


## Summary

The findings above feed directly into the **Key Business Insights** and
**Business Recommendations** sections of the project README, all grounded
in the actual numbers computed here.